# Colab Training via VS Code Extension
Make sure you have connected this notebook to a **Google Colab Kernel** using the top-right kernel selector.

### ⚠️ Action Required: Git Push
This notebook clones your code directly from a Git repository to avoid needing Google Drive. Make sure you have committed and pushed your local code to a GitHub repository before running the cell below.

In [ ]:
# 1. Clone Codebase from Git
# IMPORTANT: Replace `<YOUR_REPO_URL>` with your actual Git repository URL.
!git clone <YOUR_REPO_URL> /content/project
%cd /content/project

In [ ]:
# 2. Install dependencies
!pip install torch torchvision torchaudio -q
!pip install matplotlib numpy scipy rdkit-pypi wandb -q

In [ ]:
# 3. Authenticate with Weights & Biases (WandB)
# We use WandB to persist checkpoints instead of Google Drive.
import wandb
wandb.login()

In [ ]:
# 4. (Optional) Restore Checkpoint from WandB
# If your Colab runtime disconnected and you want to resume training:
# Uncomment the code below and replace `<YOUR_WANDB_RUN_PATH>` 
# (e.g., 'username/e3_diffusion/run-12345').
"""
import os
import wandb
api = wandb.Api()
run = api.run("<YOUR_WANDB_RUN_PATH>")
os.makedirs("/content/project/e3_diffusion_for_molecules/outputs/cfg_multi_property", exist_ok=True)
for file in run.files():
    if file.name.endswith('.npy') or file.name.endswith('.pickle'):
        file.download(root="/content/project/e3_diffusion_for_molecules/outputs/cfg_multi_property", replace=True)
print("✅ Restored checkpoint from WandB")
RESUME_FLAG="--resume outputs/cfg_multi_property"
"""
RESUME_FLAG=""  # Leave blank if starting from scratch

In [ ]:
# 5. Start Training!
!cd e3_diffusion_for_molecules && python main_qm9.py \
    --n_epochs 3000 \
    --exp_name cfg_multi_property \
    --conditioning alpha mu gap \
    --cfg_dropout_prob 0.15 \
    --cfg_dropout_mode joint \
    --n_stability_samples 500 \
    --diffusion_noise_schedule polynomial_2 \
    --diffusion_noise_precision 1e-5 \
    --diffusion_steps 1000 \
    --diffusion_loss_type l2 \
    --batch_size 32 \
    --nf 128 \
    --n_layers 6 \
    --lr 1e-4 \
    --normalize_factors "[1,4,10]" \
    --test_epochs 50 \
    --ema_decay 0.9999 \
    --save_model True \
    --wandb_usr $WANDB_USERNAME \
    {RESUME_FLAG}